# Inline 3D 检查 JSON -> Robot Renderer 变换

这个 notebook 用于：
- 读取左右 JSON 标定；
- 从 `lowdim.h5` 读取指定时间戳（可强制 exact match）的左右 joint；
- 可选对调左右 joint suffix（快速排查左右映射）；
- 从真实 depth 重建点云并做深度过滤（近/远裁剪）；
- 把点云变换到和 renderer 一致的 O3D 坐标；
- 用 `kinpy` + URDF 重建机械臂 mesh 并 inline 显示。

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import h5py
import numpy as np
import plotly.graph_objects as go
from IPython.display import display
from PIL import Image
from scipy.spatial.transform import Rotation as R

WORKSPACE_ROOT = Path('/home/haoxiang/rise2_mask_aware')
AIREXO_ROOT = WORKSPACE_ROOT / 'airexo'
for p in [WORKSPACE_ROOT, AIREXO_ROOT]:
    p_str = str(p)
    if p_str not in sys.path:
        sys.path.insert(0, p_str)

from airexo.helpers.constants import (
    O3D_RENDER_TRANSFORMATION,
    ROBOT_PREDEFINED_TRANSFORMATION,
    LEFT_ROBOT_PREDEFINED_TRANSFORMATION,
    RIGHT_ROBOT_PREDEFINED_TRANSFORMATION,
)
from airexo.helpers import urdf_robot as robot_helper
from airexo.helpers.constants import ROBOT_TCP_TO_FLANGE


In [ ]:
# ===== 用户参数 =====
LEFT_JSON = Path('/data/haoxiang/data/task0012_260321/calib/left_global_20260104/result.json')
RIGHT_JSON = Path('/data/haoxiang/data/task0012_260321/calib/right_global_20260104/result.json')
H5_PATH = Path('/data/haoxiang/data/task0012_260321/task0012_toys_basket/scene_0001/lowdim/lowdim.h5')
SCENE_DIR = Path('/data/haoxiang/data/task0012_260321/task0012_toys_basket_converted/train/scene_0001/cam_104122060902')
INTRINSICS_NPY = Path('/data/haoxiang/data/task0012_260321/1736320913189/intrinsics.npy')
INTRINSIC_KEY = 'first'

CAMERA_TIMESTAMP = 1774059832169
LEFT_ARM_SUFFIX = '062703'
RIGHT_ARM_SUFFIX = '062046'
SWAP_JOINT_SUFFIX = False   # True 时：left<-062046, right<-062703
EXACT_H5_TIMESTAMP = False  # True 时必须在 h5 timestamp 里精确命中

JOINT_SOURCE_MODE = 'teleop_init'    # 'h5' | 'teleop_init' | 'official_example'
LEFT_INIT_JOINT_DEG = [40.439804936146, -77.863036355298, -144.776131902487, 138.65111910746, 86.665128557928, -5.775515300899, -30.435855721378]
RIGHT_INIT_JOINT_DEG = [10.981961471223, -65.56937888323, -138.21245688993, 133.838226072864, 89.396671614662, -18.523809397601, -49.036677178363]
LEFT_INIT_JOINT_DEG[0] += 90
RIGHT_INIT_JOINT_DEG[0] += 90
INIT_GRIPPER_WIDTH = 0.05
# OFFICIAL_LEFT_JOINT = [1.078, -2.201, 1.628, -1.162, -0.958, 0.503, 0.875, 0.05]
# OFFICIAL_RIGHT_JOINT = [1.397, -2.226, 1.835, -1.367, -0.647, 0.781, 0.972, 0.05]

OFFICIAL_LEFT_JOINT = [ 1.774755, -2.2604105, 1.5935109, -1.8551127, -0.5085776, 0.9210934, 0.27986318, 0.00866667]
OFFICIAL_RIGHT_JOINT = [ 1.8345554, -2.2602668, 1.5923553, -1.8392365, -0.3452055, 0.97516274, 0.315274, 0.00866667]

# 0.19167139,-1.14440155,-2.41226244,2.33591771,1.56026626,-0.32330146,-0.85585147,0.00015000
# 0.70580775,-1.35896635,-2.52682018,2.41991854,1.51259184,-0.10080176,-0.53120589,0.00014000

DEPTH_SCALE = 1000.0
MIN_DEPTH_M = 0.05
MAX_DEPTH_M = 1.50
APPLY_O3D_TO_POINT_CLOUD = True

LEFT_URDF = str((WORKSPACE_ROOT / 'airexo/airexo/urdf_models/robot/left_robot_inhand.urdf').resolve())
RIGHT_URDF = str((WORKSPACE_ROOT / 'airexo/airexo/urdf_models/robot/right_robot_inhand.urdf').resolve())

SHOW_FRAME_AXES = True
SHOW_TCP_FRAME = True
FRAME_AXIS_LEN = 0.08
MESH_SAMPLE_LIMIT = 20000
POINT_STRIDE = 6
POINT_MAX = 120000


In [ ]:
class JointCfg:
    def __init__(self, num_joints=8, num_robot_joints=7):
        self.num_joints = num_joints
        self.num_robot_joints = num_robot_joints

LEFT_JOINT_CFGS = JointCfg()
RIGHT_JOINT_CFGS = JointCfg()

def invert_T(T):
    T = np.asarray(T, dtype=np.float64)
    out = np.eye(4, dtype=np.float64)
    out[:3, :3] = T[:3, :3].T
    out[:3, 3] = -T[:3, :3].T @ T[:3, 3]
    return out

def pose7_wxyz_to_mat(pose7):
    pose7 = np.asarray(pose7, dtype=np.float64).reshape(7)
    t = pose7[:3]
    qw, qx, qy, qz = pose7[3:]
    mat = np.eye(4, dtype=np.float64)
    mat[:3, :3] = R.from_quat([qx, qy, qz, qw]).as_matrix()
    mat[:3, 3] = t
    return mat

def load_json_pose(path: Path):
    data = json.loads(path.read_text())
    return pose7_wxyz_to_mat(data['pose_in_link'])

def load_intrinsic(path: Path, selector: str):
    data = np.load(str(path), allow_pickle=True)
    if isinstance(data, np.ndarray) and data.shape == ():
        data = data.item()
    if selector == 'first':
        key = sorted(data.keys())[0]
        return np.asarray(data[key], dtype=np.float64), key
    return np.asarray(data[selector], dtype=np.float64), selector

def find_h5_index(h5_timestamps: np.ndarray, camera_timestamp: int, exact: bool = False):
    if exact:
        hit = np.where(h5_timestamps == camera_timestamp)[0]
        if len(hit) == 0:
            raise ValueError(f'Exact timestamp {camera_timestamp} not found in h5')
        return int(hit[0])
    return int(np.argmin(np.abs(h5_timestamps - camera_timestamp)))

def extract_joint_and_gripper(h5_path: Path, camera_timestamp: int, left_suffix: str, right_suffix: str, exact_ts: bool):
    with h5py.File(h5_path, 'r') as f:
        h5_timestamps = np.asarray(f['timestamp'][:], dtype=np.int64)
        idx = find_h5_index(h5_timestamps, camera_timestamp, exact=exact_ts)
        matched_ts = int(h5_timestamps[idx])

        left_joint7 = np.asarray(f[f'joint_position_rad_{left_suffix}'][idx], dtype=np.float64)
        right_joint7 = np.asarray(f[f'joint_position_rad_{right_suffix}'][idx], dtype=np.float64)
        left_gripper = float(np.asarray(f[f'ee_state_{left_suffix}'][idx]).reshape(-1)[0])
        right_gripper = float(np.asarray(f[f'ee_state_{right_suffix}'][idx]).reshape(-1)[0])

        left_tcp = np.asarray(f[f'tcp_pose_{left_suffix}'][idx], dtype=np.float64)
        right_tcp = np.asarray(f[f'tcp_pose_{right_suffix}'][idx], dtype=np.float64)

    left_joint = np.concatenate([left_joint7, [left_gripper]], axis=0)
    right_joint = np.concatenate([right_joint7, [right_gripper]], axis=0)
    return idx, matched_ts, left_joint, right_joint, left_tcp, right_tcp

def init_joint_deg_to_joint(joint_deg, gripper_width):
    joint_deg = np.asarray(joint_deg, dtype=np.float64).reshape(7)
    return np.concatenate([np.deg2rad(joint_deg), [float(gripper_width)]], axis=0)

def json_base_to_cam_to_renderer_cam_to_base(T_base_to_cam):
    return invert_T(T_base_to_cam) @ invert_T(np.asarray(ROBOT_PREDEFINED_TRANSFORMATION, dtype=np.float64))

def load_mesh_vertices_faces(mesh_rel_path: str, urdf_file: str):
    import trimesh
    mesh_path = Path(urdf_file).parent / mesh_rel_path
    mesh = trimesh.load_mesh(mesh_path, process=False)
    if hasattr(mesh, 'geometry'):
        mesh = trimesh.util.concatenate(tuple(mesh.geometry.values()))
    vertices = np.asarray(mesh.vertices, dtype=np.float64)
    faces = np.asarray(mesh.faces, dtype=np.int32)
    return vertices, faces

def apply_transform(vertices: np.ndarray, T: np.ndarray):
    homo = np.concatenate([vertices, np.ones((vertices.shape[0], 1), dtype=np.float64)], axis=1)
    out = (T @ homo.T).T
    return out[:, :3]

def downsample_faces(faces: np.ndarray, limit: int):
    if faces.shape[0] <= limit:
        return faces
    idx = np.linspace(0, faces.shape[0] - 1, limit).astype(np.int64)
    return faces[idx]

def add_frame(fig, T, name, axis_len=0.08):
    origin = T[:3, 3]
    axes = T[:3, :3]
    colors = ['red', 'green', 'blue']
    labels = ['x', 'y', 'z']
    for i in range(3):
        p1 = origin
        p2 = origin + axes[:, i] * axis_len
        fig.add_trace(go.Scatter3d(
            x=[p1[0], p2[0]], y=[p1[1], p2[1]], z=[p1[2], p2[2]],
            mode='lines',
            line=dict(color=colors[i], width=6),
            name=f'{name}_{labels[i]}',
            showlegend=False,
        ))

def load_scene_images(scene_dir: Path, camera_timestamp: int):
    color_path = scene_dir / 'color' / f'{camera_timestamp}.png'
    depth_path = scene_dir / 'depth' / f'{camera_timestamp}.png'
    color = np.array(Image.open(color_path))
    depth = np.array(Image.open(depth_path))
    return color_path, depth_path, color, depth

def depth_to_point_cloud(depth_mm: np.ndarray, intrinsic: np.ndarray, stride: int, depth_scale: float, min_depth_m: float, max_depth_m: float, point_max: int, apply_o3d: bool):
    fx = intrinsic[0, 0]
    fy = intrinsic[1, 1]
    cx = intrinsic[0, 2]
    cy = intrinsic[1, 2]

    depth = depth_mm.astype(np.float64) / depth_scale
    depth = depth[::stride, ::stride]
    h, w = depth.shape
    ys, xs = np.meshgrid(np.arange(h), np.arange(w), indexing='ij')
    xs = xs * stride
    ys = ys * stride

    valid = np.isfinite(depth) & (depth > min_depth_m) & (depth < max_depth_m)
    z = depth[valid]
    x = (xs[valid] - cx) * z / fx
    y = (ys[valid] - cy) * z / fy
    pts = np.stack([x, y, z], axis=1)

    if apply_o3d:
        pts_h = np.concatenate([pts, np.ones((pts.shape[0], 1), dtype=np.float64)], axis=1)
        pts = (np.asarray(O3D_RENDER_TRANSFORMATION, dtype=np.float64) @ pts_h.T).T[:, :3]

    if pts.shape[0] > point_max:
        idx = np.linspace(0, pts.shape[0] - 1, point_max).astype(np.int64)
        pts = pts[idx]
    return pts

def safe_link_tf(tf_map, key):
    if key not in tf_map:
        return None
    return np.asarray(tf_map[key].matrix(), dtype=np.float64)

def fk_tcp_candidates(joint, joint_cfgs, urdf_file):
    tf_map = robot_helper.forward_kinematic_single(
        joint=joint.astype(np.float32),
        joint_cfgs=joint_cfgs,
        is_rad=True,
        urdf_file=urdf_file,
        with_visuals_map=False,
    )
    cand = {}
    flange = safe_link_tf(tf_map, 'flange')
    link7 = safe_link_tf(tf_map, 'link7')
    if flange is not None:
        cand['flange'] = flange
        cand['tcp_from_flange'] = flange @ invert_T(np.asarray(ROBOT_TCP_TO_FLANGE, dtype=np.float64))
    if link7 is not None:
        cand['link7'] = link7
    return cand


In [ ]:
left_suffix = LEFT_ARM_SUFFIX
right_suffix = RIGHT_ARM_SUFFIX
if SWAP_JOINT_SUFFIX:
    left_suffix, right_suffix = right_suffix, left_suffix

left_json_base_to_cam = load_json_pose(LEFT_JSON)
right_json_base_to_cam = load_json_pose(RIGHT_JSON)
left_cam_to_base = json_base_to_cam_to_renderer_cam_to_base(left_json_base_to_cam)
right_cam_to_base = json_base_to_cam_to_renderer_cam_to_base(right_json_base_to_cam)
intrinsic, intrinsic_key = load_intrinsic(INTRINSICS_NPY, INTRINSIC_KEY)

idx, matched_ts, left_joint_h5, right_joint_h5, left_tcp, right_tcp = extract_joint_and_gripper(
    H5_PATH, CAMERA_TIMESTAMP, left_suffix, right_suffix, EXACT_H5_TIMESTAMP
)

left_joint_init = init_joint_deg_to_joint(LEFT_INIT_JOINT_DEG, INIT_GRIPPER_WIDTH)
right_joint_init = init_joint_deg_to_joint(RIGHT_INIT_JOINT_DEG, INIT_GRIPPER_WIDTH)
left_joint_official = np.asarray(OFFICIAL_LEFT_JOINT, dtype=np.float64)
right_joint_official = np.asarray(OFFICIAL_RIGHT_JOINT, dtype=np.float64)

if JOINT_SOURCE_MODE == 'teleop_init':
    left_joint = left_joint_init.copy()
    right_joint = right_joint_init.copy()
elif JOINT_SOURCE_MODE == 'official_example':
    left_joint = left_joint_official.copy()
    right_joint = right_joint_official.copy()
else:
    left_joint = left_joint_h5.copy()
    right_joint = right_joint_h5.copy()

color_path, depth_path, color_img, depth_img = load_scene_images(SCENE_DIR, CAMERA_TIMESTAMP)
point_cloud = depth_to_point_cloud(
    depth_img, intrinsic, stride=POINT_STRIDE, depth_scale=DEPTH_SCALE,
    min_depth_m=MIN_DEPTH_M, max_depth_m=MAX_DEPTH_M, point_max=POINT_MAX,
    apply_o3d=APPLY_O3D_TO_POINT_CLOUD
)

print('camera_timestamp      =', CAMERA_TIMESTAMP)
print('matched_h5_index      =', idx)
print('matched_h5_timestamp  =', matched_ts)
print('timestamp_diff        =', abs(matched_ts - CAMERA_TIMESTAMP))
print('exact_h5_timestamp    =', EXACT_H5_TIMESTAMP)
print('joint_source_mode     =', JOINT_SOURCE_MODE)
print('swap_joint_suffix     =', SWAP_JOINT_SUFFIX)
print('left_suffix used      =', left_suffix)
print('right_suffix used     =', right_suffix)
print('intrinsic_key         =', intrinsic_key)
print('min_depth_m           =', MIN_DEPTH_M)
print('max_depth_m           =', MAX_DEPTH_M)
print('apply_o3d_to_pcd      =', APPLY_O3D_TO_POINT_CLOUD)
print('point_cloud shape     =', point_cloud.shape)
print('left_joint_h5         =', left_joint_h5)
print('right_joint_h5        =', right_joint_h5)
print('left_joint_init       =', left_joint_init)
print('right_joint_init      =', right_joint_init)
print('left_joint_official   =', left_joint_official)
print('right_joint_official  =', right_joint_official)
print('left_joint_used       =', left_joint)
print('right_joint_used      =', right_joint)
print('left_tcp (h5)         =', left_tcp)
print('right_tcp (h5)        =', right_tcp)
print('left_cam_to_base =\n', left_cam_to_base)
print('right_cam_to_base =\n', right_cam_to_base)


In [ ]:
cur_transforms_left, visuals_map_left = robot_helper.forward_kinematic_single(
    joint=left_joint.astype(np.float32),
    joint_cfgs=LEFT_JOINT_CFGS,
    is_rad=True,
    urdf_file=LEFT_URDF,
    with_visuals_map=True,
)
cur_transforms_right, visuals_map_right = robot_helper.forward_kinematic_single(
    joint=right_joint.astype(np.float32),
    joint_cfgs=RIGHT_JOINT_CFGS,
    is_rad=True,
    urdf_file=RIGHT_URDF,
    with_visuals_map=True,
)
left_fk_candidates = fk_tcp_candidates(left_joint, LEFT_JOINT_CFGS, LEFT_URDF)
right_fk_candidates = fk_tcp_candidates(right_joint, RIGHT_JOINT_CFGS, RIGHT_URDF)
print('left links :', len(cur_transforms_left))
print('right links:', len(cur_transforms_right))
print('left fk candidates :', list(left_fk_candidates.keys()))
print('right fk candidates:', list(right_fk_candidates.keys()))
for name, T in left_fk_candidates.items():
    print(f'left::{name} =\n', T)
for name, T in right_fk_candidates.items():
    print(f'right::{name} =\n', T)


In [ ]:
fig = go.Figure()

if point_cloud.shape[0] > 0:
    fig.add_trace(go.Scatter3d(
        x=point_cloud[:, 0],
        y=point_cloud[:, 1],
        z=point_cloud[:, 2],
        mode='markers',
        marker=dict(size=1.0, color=point_cloud[:, 2], colorscale='Viridis', opacity=0.35),
        name='depth_point_cloud',
    ))

left_color = 'rgba(50,120,255,0.65)'
right_color = 'rgba(255,120,50,0.65)'

for link, transform in cur_transforms_left.items():
    for v in visuals_map_left[link]:
        if v.geom_param is None:
            continue
        verts, faces = load_mesh_vertices_faces(v.geom_param, LEFT_URDF)
        faces = downsample_faces(faces, MESH_SAMPLE_LIMIT)
        tf = (
            np.asarray(O3D_RENDER_TRANSFORMATION, dtype=np.float64)
            @ left_cam_to_base
            @ np.asarray(ROBOT_PREDEFINED_TRANSFORMATION, dtype=np.float64)
            @ np.asarray(LEFT_ROBOT_PREDEFINED_TRANSFORMATION, dtype=np.float64)
            @ transform.matrix()
            @ v.offset.matrix()
        )
        verts_tf = apply_transform(verts, tf)
        fig.add_trace(go.Mesh3d(
            x=verts_tf[:, 0],
            y=verts_tf[:, 1],
            z=verts_tf[:, 2],
            i=faces[:, 0],
            j=faces[:, 1],
            k=faces[:, 2],
            color=left_color,
            opacity=0.55,
            name=f'left::{link}',
            showscale=False,
            hoverinfo='name',
        ))

for link, transform in cur_transforms_right.items():
    for v in visuals_map_right[link]:
        if v.geom_param is None:
            continue
        verts, faces = load_mesh_vertices_faces(v.geom_param, RIGHT_URDF)
        faces = downsample_faces(faces, MESH_SAMPLE_LIMIT)
        tf = (
            np.asarray(O3D_RENDER_TRANSFORMATION, dtype=np.float64)
            @ right_cam_to_base
            @ np.asarray(ROBOT_PREDEFINED_TRANSFORMATION, dtype=np.float64)
            @ np.asarray(RIGHT_ROBOT_PREDEFINED_TRANSFORMATION, dtype=np.float64)
            @ transform.matrix()
            @ v.offset.matrix()
        )
        verts_tf = apply_transform(verts, tf)
        fig.add_trace(go.Mesh3d(
            x=verts_tf[:, 0],
            y=verts_tf[:, 1],
            z=verts_tf[:, 2],
            i=faces[:, 0],
            j=faces[:, 1],
            k=faces[:, 2],
            color=right_color,
            opacity=0.55,
            name=f'right::{link}',
            showscale=False,
            hoverinfo='name',
        ))

if SHOW_FRAME_AXES:
    add_frame(fig, np.eye(4), 'camera', axis_len=FRAME_AXIS_LEN)
    left_base_world = (
        np.asarray(O3D_RENDER_TRANSFORMATION, dtype=np.float64)
        @ left_cam_to_base
        @ np.asarray(ROBOT_PREDEFINED_TRANSFORMATION, dtype=np.float64)
        @ np.asarray(LEFT_ROBOT_PREDEFINED_TRANSFORMATION, dtype=np.float64)
    )
    right_base_world = (
        np.asarray(O3D_RENDER_TRANSFORMATION, dtype=np.float64)
        @ right_cam_to_base
        @ np.asarray(ROBOT_PREDEFINED_TRANSFORMATION, dtype=np.float64)
        @ np.asarray(RIGHT_ROBOT_PREDEFINED_TRANSFORMATION, dtype=np.float64)
    )
    add_frame(fig, left_base_world, 'left_base', axis_len=FRAME_AXIS_LEN)
    add_frame(fig, right_base_world, 'right_base', axis_len=FRAME_AXIS_LEN)
    if SHOW_TCP_FRAME and "tcp_from_flange" in left_fk_candidates:
        left_tcp_world = (
            np.asarray(O3D_RENDER_TRANSFORMATION, dtype=np.float64)
            @ left_cam_to_base
            @ np.asarray(ROBOT_PREDEFINED_TRANSFORMATION, dtype=np.float64)
            @ np.asarray(LEFT_ROBOT_PREDEFINED_TRANSFORMATION, dtype=np.float64)
            @ left_fk_candidates["tcp_from_flange"]
        )
        add_frame(fig, left_tcp_world, 'left_tcp_fk', axis_len=FRAME_AXIS_LEN * 0.8)
    if SHOW_TCP_FRAME and "tcp_from_flange" in right_fk_candidates:
        right_tcp_world = (
            np.asarray(O3D_RENDER_TRANSFORMATION, dtype=np.float64)
            @ right_cam_to_base
            @ np.asarray(ROBOT_PREDEFINED_TRANSFORMATION, dtype=np.float64)
            @ np.asarray(RIGHT_ROBOT_PREDEFINED_TRANSFORMATION, dtype=np.float64)
            @ right_fk_candidates["tcp_from_flange"]
        )
        add_frame(fig, right_tcp_world, 'right_tcp_fk', axis_len=FRAME_AXIS_LEN * 0.8)

fig.update_layout(
    title='Inline 3D: Filtered Depth Point Cloud + Robot Mesh',
    scene=dict(
        xaxis_title='X',
        yaxis_title='Y',
        zaxis_title='Z',
        aspectmode='data',
    ),
    width=1300,
    height=950,
    showlegend=False,
)
display(fig)


In [ ]:
def render_inline_compare(left_joint_used: np.ndarray, right_joint_used: np.ndarray, title: str):
    cur_transforms_left_cmp, visuals_map_left_cmp = robot_helper.forward_kinematic_single(
        joint=left_joint_used.astype(np.float32),
        joint_cfgs=LEFT_JOINT_CFGS,
        is_rad=True,
        urdf_file=LEFT_URDF,
        with_visuals_map=True,
    )
    cur_transforms_right_cmp, visuals_map_right_cmp = robot_helper.forward_kinematic_single(
        joint=right_joint_used.astype(np.float32),
        joint_cfgs=RIGHT_JOINT_CFGS,
        is_rad=True,
        urdf_file=RIGHT_URDF,
        with_visuals_map=True,
    )

    fig_cmp = go.Figure()
    if point_cloud.shape[0] > 0:
        fig_cmp.add_trace(go.Scatter3d(
            x=point_cloud[:, 0], y=point_cloud[:, 1], z=point_cloud[:, 2],
            mode='markers',
            marker=dict(size=1.0, color=point_cloud[:, 2], colorscale='Viridis', opacity=0.35),
            name='depth_point_cloud',
            showlegend=False,
        ))

    left_color_cmp = 'rgba(50,120,255,0.65)'
    right_color_cmp = 'rgba(255,120,50,0.65)'

    for link, transform in cur_transforms_left_cmp.items():
        for v in visuals_map_left_cmp[link]:
            if v.geom_param is None:
                continue
            verts, faces = load_mesh_vertices_faces(v.geom_param, LEFT_URDF)
            faces = downsample_faces(faces, MESH_SAMPLE_LIMIT)
            tf = (
                np.asarray(O3D_RENDER_TRANSFORMATION, dtype=np.float64)
                @ left_cam_to_base
                @ np.asarray(ROBOT_PREDEFINED_TRANSFORMATION, dtype=np.float64)
                @ np.asarray(LEFT_ROBOT_PREDEFINED_TRANSFORMATION, dtype=np.float64)
                @ transform.matrix()
                @ v.offset.matrix()
            )
            verts_tf = apply_transform(verts, tf)
            fig_cmp.add_trace(go.Mesh3d(
                x=verts_tf[:, 0], y=verts_tf[:, 1], z=verts_tf[:, 2],
                i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
                color=left_color_cmp, opacity=0.55,
                showscale=False, hoverinfo='skip', showlegend=False,
            ))

    for link, transform in cur_transforms_right_cmp.items():
        for v in visuals_map_right_cmp[link]:
            if v.geom_param is None:
                continue
            verts, faces = load_mesh_vertices_faces(v.geom_param, RIGHT_URDF)
            faces = downsample_faces(faces, MESH_SAMPLE_LIMIT)
            tf = (
                np.asarray(O3D_RENDER_TRANSFORMATION, dtype=np.float64)
                @ right_cam_to_base
                @ np.asarray(ROBOT_PREDEFINED_TRANSFORMATION, dtype=np.float64)
                @ np.asarray(RIGHT_ROBOT_PREDEFINED_TRANSFORMATION, dtype=np.float64)
                @ transform.matrix()
                @ v.offset.matrix()
            )
            verts_tf = apply_transform(verts, tf)
            fig_cmp.add_trace(go.Mesh3d(
                x=verts_tf[:, 0], y=verts_tf[:, 1], z=verts_tf[:, 2],
                i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
                color=right_color_cmp, opacity=0.55,
                showscale=False, hoverinfo='skip', showlegend=False,
            ))

    if SHOW_FRAME_AXES:
        add_frame(fig_cmp, np.eye(4), 'camera', axis_len=FRAME_AXIS_LEN)
        add_frame(fig_cmp, left_cam_to_base, 'left_base', axis_len=FRAME_AXIS_LEN)
        add_frame(fig_cmp, right_cam_to_base, 'right_base', axis_len=FRAME_AXIS_LEN)

    fig_cmp.update_layout(
        title=title,
        scene=dict(xaxis_title='X', yaxis_title='Y', zaxis_title='Z', aspectmode='data'),
        width=1300,
        height=950,
        showlegend=False,
    )
    display(fig_cmp)


In [ ]:
# 对比 Cell 1：官方示例 joint
left_joint_cmp = np.asarray(OFFICIAL_LEFT_JOINT, dtype=np.float64)
right_joint_cmp = np.asarray(OFFICIAL_RIGHT_JOINT, dtype=np.float64)
print('compare mode = official_example')
print('left_joint_cmp =', left_joint_cmp)
print('right_joint_cmp =', right_joint_cmp)
render_inline_compare(left_joint_cmp, right_joint_cmp, title='Compare 1/2: Official Example Joint')


In [ ]:
# 对比 Cell 2：你的 h5 joint（按当前 suffix 映射）
left_joint_cmp = left_joint_h5.copy()
right_joint_cmp = right_joint_h5.copy()
left_joint_cmp[0] += 1.5707963267948966  # 90度
right_joint_cmp[0] += 1.5707963267948966  # 90度

print('compare mode = our_h5_joint')
print('left_joint_cmp =', left_joint_cmp)
print('right_joint_cmp =', right_joint_cmp)
render_inline_compare(left_joint_cmp, right_joint_cmp, title='Compare 2/2: Our H5 Joint')
